In [1]:
import numpy as np
from meridian.optimizer.independent_optimizer import OptimizationInput, \
   ModelParameters, IndependentOptimizer, OptimizationScenario

from meridian.optimizer import optimization_core

1. Create Mock Data

In [2]:
# Example channels
impression_channels = ['tv', 'digital', 'radio']
rf_channels = ['social_media', 'display_video']

# Example dimensions
n_geos = 3
n_times = 52  # 52 weeks
n_impression_channels = len(impression_channels)
n_rf_channels = len(rf_channels)

# Generate synthetic historical spend
impression_historical_spend = np.array([100000, 80000, 50000])  # $230k total
rf_historical_spend = np.array([40000, 30000])  # $70k total

# Generate synthetic population data
geos = ['US_West', 'US_Central', 'US_East']
population = np.array([50000000, 65000000, 75000000])  # Population per geo


In [3]:
# Generate synthetic impression data (n_geos, n_times, n_impression_channels)
np.random.seed(42)
impression_data = np.random.lognormal(
    mean=8, sigma=1, size=(n_geos, n_times, n_impression_channels)
)

In [4]:
# Generate synthetic R&F data
# Reach: number of unique people reached
rf_reach = np.random.lognormal(
    mean=6, sigma=0.8, size=(n_geos, n_times, n_rf_channels)
)

# Frequency: average exposures per person (typically 1-10)
rf_frequency = np.random.lognormal(
    mean=1, sigma=0.5, size=(n_geos, n_times, n_rf_channels)
)
rf_frequency = np.clip(rf_frequency, 1.0, 10.0)  # Realistic frequency range

# R&F spend data
rf_spend = np.random.lognormal(
    mean=7, sigma=0.8, size=(n_geos, n_times, n_rf_channels)
)

In [5]:
# Generate dates
import pandas as pd
dates = pd.date_range('2024-01-01', periods=n_times, freq='W').strftime('%Y-%m-%d').tolist()


In [6]:
# Create optimization input with mixed channels
input_data = OptimizationInput(
    impression_channels=impression_channels,
    rf_channels=rf_channels,
    impression_historical_spend=impression_historical_spend,
    rf_historical_spend=rf_historical_spend,
    impression_data=impression_data,
    rf_reach=rf_reach,
    rf_frequency=rf_frequency,
    rf_spend=rf_spend,
    dates=dates,
    geos=geos,
    population=population,
)

In [9]:
input_data.impression_data.shape

(3, 52, 3)

2. Initialize Model Params

In [10]:
# Generate synthetic model parameters for mixed channels
model_params = ModelParameters(
    # Impression channel parameters
    impression_coefficients=np.random.uniform(0.5, 2.0, size=(n_geos, n_impression_channels)),
    impression_adstock_params=np.random.uniform(0.1, 0.7, size=n_impression_channels),
    impression_ec50_params=np.random.uniform(0.3, 0.8, size=n_impression_channels),
    impression_slope_params=np.random.uniform(0.8, 2.0, size=n_impression_channels),
    # R&F channel parameters
    rf_coefficients=np.random.uniform(0.3, 1.5, size=(n_geos, n_rf_channels)),
    rf_adstock_params=np.random.uniform(0.2, 0.6, size=n_rf_channels),
    rf_ec50_params=np.random.uniform(2.0, 6.0, size=n_rf_channels),  # Higher EC50 for frequency
    rf_slope_params=np.random.uniform(1.0, 2.5, size=n_rf_channels),
    # Common parameters
    baseline=np.random.uniform(1000, 5000, size=n_geos),
)


3. Initialize Optimizer

In [11]:
# Initialize optimizer
optimizer = IndependentOptimizer(input_data, model_params)

# Define optimization scenario
scenario = OptimizationScenario(
    scenario_type='fixed_budget',
    total_budget=320000,  # 6.7% increase from $300k historical (230k impression + 70k R&F)
    spend_constraint_lower=0.2,  # Allow 20% decrease
    spend_constraint_upper=0.4,  # Allow 40% increase
)

In [12]:
results = optimizer.optimize(scenario)

In [13]:
results

IndependentOptimizationResults(optimized_spend=array([104533,  71146,  46186,  53333,  44800]), historical_spend=array([100000,  80000,  50000,  40000,  30000]), incremental_outcome=array([158.38121,  97.52991, 131.25696, 102.06573, 188.22125],
      dtype=float32), roi_by_channel=array([0.00151513, 0.00137084, 0.00284192, 0.00191374, 0.00420137],
      dtype=float32), total_roi=np.float64(0.002117060350767817), scenario=OptimizationScenario(scenario_type='fixed_budget', total_budget=320000, target_roi=None, target_mroi=None, spend_constraint_lower=0.2, spend_constraint_upper=0.4, start_date=None, end_date=None), channels=['tv', 'digital', 'radio', 'social_media', 'display_video'])

In [26]:
# # Run optimization
# results = optimizer.optimize(scenario)

# Determine optimization period
self = optimizer
selected_times = self._get_time_slice(scenario.start_date, scenario.end_date)

# Set up spend constraints using combined historical spend
all_historical_spend = self.input_data.all_historical_spend
budget = scenario.total_budget or np.sum(all_historical_spend)
spend_allocation = budget * (all_historical_spend / np.sum(all_historical_spend))


In [30]:
lower_bounds, upper_bounds = optimization_core.validate_spend_constraints(
    spend_allocation=spend_allocation,
    historical_spend=all_historical_spend,
    spend_constraint_lower=scenario.spend_constraint_lower,
    spend_constraint_upper=scenario.spend_constraint_upper,
)

In [34]:
spend_allocation * (1 - 0.2), spend_allocation * (1 + 0.4)

(array([85333.33333333, 68266.66666667, 42666.66666667, 34133.33333333,
        25600.        ]),
 array([149333.33333333, 119466.66666667,  74666.66666667,  59733.33333333,
         44800.        ]))

In [35]:
lower_bounds, upper_bounds

(array([85333.33333333, 68266.66666667, 42666.66666667, 34133.33333333,
        25600.        ]),
 array([149333.33333333, 119466.66666667,  74666.66666667,  59733.33333333,
         44800.        ]))

In [ ]:
spend_allocation / all_historical_spend

array([1.06666667, 1.06666667, 1.06666667, 1.06666667, 1.06666667])